In [7]:
import os
import nrrd
import numpy as np
import nrrd
import SimpleITK as sitk
from AIAH_utility.viewer import BasicViewer
from tqdm import tqdm
import re
import pandas as pd
import json

### DICOM to NRRD

In [29]:
data_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/manifest-1777460989222/prostate_mri_us_biopsy'
out_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/'
print(len(os.listdir(data_dir)))
'''
for file in os.listdir(data_dir):
    if not os.path.isfile(os.path.join(data_dir, file)):
        if len(os.listdir(os.path.join(data_dir, file))) > 1:
            print(file)
'''  

353


'\nfor file in os.listdir(data_dir):\n    if not os.path.isfile(os.path.join(data_dir, file)):\n        if len(os.listdir(os.path.join(data_dir, file))) > 1:\n            print(file)\n'

In [18]:


def identify_mri_sequence(dicom_dir):
    # 1. Get all DICOM files in the folder
    reader = sitk.ImageSeriesReader()
    series_IDs = reader.GetGDCMSeriesIDs(dicom_dir)
    
    if not series_IDs:
        print("No DICOM files found.")
        return None

    # Get the filenames for the first series found
    dicom_names = reader.GetGDCMSeriesFileNames(dicom_dir, series_IDs[0])
    first_file = dicom_names[0] # We only need to check the first slice's header

    # 2. Read only the header information (fast, doesn't load image pixels)
    file_reader = sitk.ImageFileReader()
    file_reader.SetFileName(first_file)
    file_reader.ReadImageInformation()

    # 3. Extract the Series Description (DICOM Tag: 0008,103E)
    # Using a try/except because occasionally some anonymized DICOMs strip this tag
    try:
        series_description = file_reader.GetMetaData("0008|103e").lower()
    except RuntimeError:
        series_description = "unknown (tag missing)"

    #print(f"Raw DICOM Series Description: '{series_description}'")

    # 4. Keyword matching to identify the sequence
    sequence_type = "Unclassified"
    desc_lower = series_description.lower()
    dwi_keywords = ["dwi", "calc_bval", "ax", "bval"]

    if "t2" in desc_lower:
        sequence_type = "t2"
    elif "adc" in desc_lower:
        sequence_type = "adc"
    elif any(word in desc_lower for word in dwi_keywords) or re.search(r'b\d+', desc_lower):
        sequence_type = "dwi"

        
    #print(f"Identified Sequence Type: {sequence_type}")
    return sequence_type, series_description



def dicom_to_nrrd(dicom_dir, output_filepath):
    # 1. Initialize the ImageSeriesReader
    reader = sitk.ImageSeriesReader()

    # 2. Find the DICOM series within the folder
    # A single folder might contain multiple scans (e.g., a T1 and a T2). 
    # This function groups them by their unique Series Instance UID.
    series_IDs = reader.GetGDCMSeriesIDs(dicom_dir)
    
    if not series_IDs:
        print(f"Error: No DICOM series found in the directory {dicom_dir}.")
        return

    # Assuming you want the first (or only) series in the folder
    series_ID = series_IDs[0]
    
    # 3. Get the list of files belonging to this specific series.
    # SimpleITK automatically sorts them by spatial position/instance number here.
    dicom_names = reader.GetGDCMSeriesFileNames(dicom_dir, series_ID)
    reader.SetFileNames(dicom_names)

    #print(f"Found series {series_ID}")
    #print(f"Reading {len(dicom_names)} DICOM files...")

    # 4. Read the files into a single 3D image object
    image = reader.Execute()
    '''
    # (Optional) Print out the header info that SimpleITK extracted
    print("\nExtracted Header Information:")
    print(f"Size (X, Y, Z): {image.GetSize()}")
    print(f"Spacing (X, Y, Z): {image.GetSpacing()}")
    print(f"Origin (X, Y, Z): {image.GetOrigin()}")
    print(f"Direction Cosine Matrix: {image.GetDirection()}\n")

    # 5. Write the 3D image to an NRRD file
    # SimpleITK automatically translates the spatial metadata into the NRRD header format.
    '''
    sitk.WriteImage(image, output_filepath)
    #print(f"Successfully saved 3D volume to: {output_filepath}")






In [19]:
import logging


# Keep your basic logging config
logging.basicConfig(
    filename='dicom_warnings.log',
    filemode='a',
    level=logging.WARNING,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

class SimpleITKWarningTracker(sitk.LoggerBase):
    def __init__(self, logger: logging.Logger = logging.getLogger("SimpleITK")):
        super().__init__()
        self._logger = logger
        
        # --- NEW ADDITIONS ---
        self.current_uid = None
        self.problematic_uids = set() # Using a set so we don't get duplicates

    def DisplayWarningText(self, s):
        # 1. Log to the file, but inject the UID so you know exactly which one failed
        self._logger.warning(f"[{self.current_uid}] {s.rstrip()}")
        
        # 2. Capture the UID into our set
        if self.current_uid is not None:
            self.problematic_uids.add(self.current_uid)

    def DisplayErrorText(self, s):
        self._logger.error(f"[{self.current_uid}] {s.rstrip()}")
        if self.current_uid is not None:
            self.problematic_uids.add(self.current_uid)

    # Standard passthroughs for everything else
    def DisplayText(self, s): self._logger.info(s.rstrip())
    def DisplayGenericOutputText(self, s): self._logger.info(s.rstrip())
    def DisplayDebugText(self, s): self._logger.debug(s.rstrip())

In [32]:



def get_clean_file_set(folder_path):
    """
    Returns a set of filenames in a folder, 
    ignoring hidden files like .DS_Store.
    """
    if not os.path.exists(folder_path):
        print(f"Warning: Folder not found -> {folder_path}")
        return set()
        
    return set(f for f in os.listdir(folder_path) if not f.startswith('.'))

def audit_mri_folders(t2_dir, dwi_dir, adc_dir):
    # 1. Grab the sets of files from each folder
    t2_files = get_clean_file_set(t2_dir)
    dwi_files = get_clean_file_set(dwi_dir)
    adc_files = get_clean_file_set(adc_dir)
    
    # 2. Check for absolute perfection
    if t2_files == dwi_files == adc_files:
        print("✅ SUCCESS: All three folders contain the exact same files.")
        print(f"Total files per folder: {len(t2_files)}")
        return

    # 3. If they don't match, figure out exactly what went wrong
    print("❌ MISMATCH DETECTED: The folders do not have the same files.\n")
    
    # Create a master list of EVERY unique file found across all three folders
    all_known_files = t2_files | dwi_files | adc_files
    print(f"Total unique cases found across all folders: {len(all_known_files)}\n")
    
    # Subtracting a folder's files from the master list reveals exactly what it is missing
    missing_from_t2 = all_known_files - t2_files
    missing_from_dwi = all_known_files - dwi_files
    missing_from_adc = all_known_files - adc_files
    
    # 4. Print the detailed report
    if missing_from_t2:
        print(f"Missing from T2 ({len(missing_from_t2)} files):")
        for f in missing_from_t2: print(f"  - {f}")
        print()
        
    if missing_from_dwi:
        print(f"Missing from DWI ({len(missing_from_dwi)} files):")
        for f in missing_from_dwi: print(f"  - {f}")
        print()
        
    if missing_from_adc:
        print(f"Missing from ADC ({len(missing_from_adc)} files):")
        for f in missing_from_adc: print(f"  - {f}")
        print()

# ==========================================
# Run the auditor
# ==========================================

t2_folder = os.path.join(out_dir, "t2")
adc_folder = os.path.join(out_dir, "adc")
dwi_folder = os.path.join(out_dir, "dwi")
audit_mri_folders(t2_folder, dwi_folder, adc_folder)


✅ SUCCESS: All three folders contain the exact same files.
Total files per folder: 336


In [28]:

def get_clean_file_set(folder_path):
    if not os.path.exists(folder_path):
        return set()
    return set(f for f in os.listdir(folder_path) if not f.startswith('.'))

def remove_orphan_files(t2_dir, dwi_dir, adc_dir, dry_run=True):
    # 1. Grab the sets of files from each folder
    t2_files = get_clean_file_set(t2_dir)
    dwi_files = get_clean_file_set(dwi_dir)
    adc_files = get_clean_file_set(adc_dir)
    
    # 2. Find the "Perfect Matches" (files that exist in ALL three folders)
    common_files = t2_files & dwi_files & adc_files
    
    # 3. Setup directories to check
    directories = [
        ("T2", t2_dir, t2_files),
        ("DWI", dwi_dir, dwi_files),
        ("ADC", adc_dir, adc_files)
    ]
    
    total_deleted = 0
    
    print(f"--- Running in {'DRY RUN (Safe)' if dry_run else 'ACTIVE DELETE'} mode ---\n")
    
    # 4. Loop through each folder and delete files that aren't in the common pool
    for name, dir_path, files in directories:
        orphans = files - common_files
        
        if not orphans:
            print(f"✅ {name} folder is already clean.")
            continue
            
        print(f"🧹 Cleaning {name} folder ({len(orphans)} orphan files found)...")
        for orphan in orphans:
            file_path = os.path.join(dir_path, orphan)
            
            if dry_run:
                print(f"   [Would Delete] -> {file_path}")
            else:
                try:
                    os.remove(file_path)
                    print(f"   [Deleted] -> {file_path}")
                    total_deleted += 1
                except Exception as e:
                    print(f"   [Error] -> Could not delete {file_path}: {e}")
                    
    # 5. Final Summary
    print("\n" + "="*40)
    if dry_run:
        print("🛑 This was a DRY RUN. No files were actually deleted.")
        print("To permanently delete these files, change 'dry_run=False' in the script.")
    else:
        print(f"✅ Cleanup complete. {total_deleted} orphan files permanently deleted.")
    print("="*40)



# Run it once with True to see what will happen.
# Change to False when you are ready to delete.
remove_orphan_files(t2_folder, dwi_folder, adc_folder, dry_run=False)

--- Running in ACTIVE DELETE mode ---

🧹 Cleaning T2 folder (6 orphan files found)...
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.115783195095504090298691430241561836887.nrrd
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.73817330564297277045402819763812121589.nrrd
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.57998954095255348100273416607857947941.nrrd
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.71945175083621641395573395567726460544.nrrd
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.70790563074707380958612618193727014402.nrrd
   [Deleted] -> /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.997433357123347623636472062

In [31]:
import os

def delete_bad_uids(t2_dir, dwi_dir, adc_dir, bad_uids_list, dry_run=True):
    # The folders we need to check
    directories = {
        "T2": t2_dir,
        "DWI": dwi_dir,
        "ADC": adc_dir
    }
    
    total_deleted = 0
    total_not_found = 0
    
    print(f"--- Running in {'DRY RUN (Safe)' if dry_run else 'ACTIVE DELETE'} mode ---\n")
    
    # Loop through every bad UID in your list
    for uid in bad_uids_list:
        filename = f"{uid}.nrrd"
        print(f"Targeting: {filename}")
        
        # Check all three folders for this specific file
        for folder_name, folder_path in directories.items():
            file_path = os.path.join(folder_path, filename)
            
            if os.path.exists(file_path):
                if dry_run:
                    print(f"   [{folder_name}] -> [Would Delete]: {file_path}")
                else:
                    try:
                        os.remove(file_path)
                        print(f"   [{folder_name}] -> [Deleted]: {file_path}")
                        total_deleted += 1
                    except Exception as e:
                        print(f"   [{folder_name}] -> [Error deleting]: {file_path} - {e}")
            else:
                # Optional: Uncomment the line below if you want to know when a file was already missing
                # print(f"   [{folder_name}] -> Not found (already gone)")
                total_not_found += 1
                
    # Final Summary
    print("\n" + "="*40)
    if dry_run:
        print("🛑 This was a DRY RUN. No files were actually deleted.")
        print("To permanently delete these files, change 'dry_run=False' in the script.")
    else:
        print(f"✅ Cleanup complete. {total_deleted} files permanently deleted.")
    print("="*40)



# Paste your list of bad UIDs here
# (e.g., from the 'bad_uids = list(tracker.problematic_uids)' step earlier)

bad_uids = list(tracker.problematic_uids)

# Run it once with True to verify. 
# Change to False to actually delete the files.
delete_bad_uids(t2_folder, dwi_folder, adc_folder, bad_uids, dry_run=False)

--- Running in ACTIVE DELETE mode ---

Targeting: 1.3.6.1.4.1.14519.5.2.1.271349975011933739358676771067980247539.nrrd
   [T2] -> [Deleted]: /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.271349975011933739358676771067980247539.nrrd
   [DWI] -> [Deleted]: /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/dwi/1.3.6.1.4.1.14519.5.2.1.271349975011933739358676771067980247539.nrrd
   [ADC] -> [Deleted]: /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/adc/1.3.6.1.4.1.14519.5.2.1.271349975011933739358676771067980247539.nrrd
Targeting: 1.3.6.1.4.1.14519.5.2.1.157054200734545334178920696903308865130.nrrd
   [T2] -> [Deleted]: /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/t2/1.3.6.1.4.1.14519.5.2.1.157054200734545334178920696903308865130.nrrd
   [DWI] -> [Deleted]: /sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/dwi/1.3.6.1.4.1.14519.5.2.1.157054200734545334178920696903

In [26]:
tracker = SimpleITKWarningTracker()
tracker.SetAsGlobalITKLogger()
for file in tqdm(os.listdir(data_dir)):
    if not os.path.isfile(os.path.join(data_dir, file)):
        for st_uid in os.listdir(os.path.join(data_dir, file)):
            tracker.current_uid = st_uid
            for seq in os.listdir(os.path.join(data_dir, file, st_uid)):
                seq_path = os.path.join(data_dir, file, st_uid, seq)
                seq_type, _ = identify_mri_sequence(seq_path)
                output_nrrd_path = os.path.join(out_dir,seq_type, f"{st_uid}.nrrd")       

                dicom_to_nrrd(seq_path, output_nrrd_path)
                bool_ = True

                
                
bad_uids = list(tracker.problematic_uids)
for uid in bad_uids:
    print(uid)      


'''

st_uid = os.listdir(os.path.join(data_dir, file))[0]
print(st_uid)
seqs = os.listdir(os.path.join(data_dir, file, st_uid))
seqs = os.listdir(os.path.join(data_dir, file, st_uid))
print(seqs)
seq = os.path.join(data_dir, file, st_uid, seqs[1])
len(os.listdir(seq))
'''

100%|██████████| 353/353 [12:23<00:00,  2.11s/it]

1.3.6.1.4.1.14519.5.2.1.271349975011933739358676771067980247539
1.3.6.1.4.1.14519.5.2.1.157054200734545334178920696903308865130
1.3.6.1.4.1.14519.5.2.1.200600558595197166260928361613464269449
1.3.6.1.4.1.14519.5.2.1.289285689423372767351515144958159282808
1.3.6.1.4.1.14519.5.2.1.35974516785743444930073912740842262500
1.3.6.1.4.1.14519.5.2.1.222692088368162168440753904825013091532
1.3.6.1.4.1.14519.5.2.1.293474077433499282588478960568109869597
1.3.6.1.4.1.14519.5.2.1.27532522792473057810687373197700121309
1.3.6.1.4.1.14519.5.2.1.239133116807588836296851847147700495752
1.3.6.1.4.1.14519.5.2.1.227398582376562514609189345768801704404
1.3.6.1.4.1.14519.5.2.1.17858146857447950947031599871374548190
1.3.6.1.4.1.14519.5.2.1.58794500493878684077452076491488634390
1.3.6.1.4.1.14519.5.2.1.81532906654688928393462620176242817061
1.3.6.1.4.1.14519.5.2.1.191423027298795057608214686303929880649
1.3.6.1.4.1.14519.5.2.1.126258218767968455484960185246643285805
1.3.6.1.4.1.14519.5.2.1.269756171143806913587

'\n\nst_uid = os.listdir(os.path.join(data_dir, file))[0]\nprint(st_uid)\nseqs = os.listdir(os.path.join(data_dir, file, st_uid))\nseqs = os.listdir(os.path.join(data_dir, file, st_uid))\nprint(seqs)\nseq = os.path.join(data_dir, file, st_uid, seqs[1])\nlen(os.listdir(seq))\n'

### Extract JSON

In [2]:
data_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/'
meta_folder = "/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/manifest-1777460989222/metadata"
t2_dir = os.path.join(data_dir, "t2")
adc_dir = os.path.join(data_dir, "adc")
dwi_dir = os.path.join(data_dir, "dwi")
print(os.listdir(t2_dir) == os.listdir(adc_dir) == os.listdir(dwi_dir))

True


In [3]:
st_id = os.listdir(t2_dir)

df_parent = pd.read_excel(os.path.join(meta_folder, "Prostate-MRI-US-Biopsy-NBIA-manifest_v2_20231020-nbia-digest.xlsx"))
df = pd.read_excel(os.path.join(meta_folder, "TCIA-Biopsy-Data_2020-07-14.xlsx"))


In [4]:
exclude = []
test_list = []
for file in st_id:
    
    id = file.split(".nrrd")[0]
    filtered_parent = df_parent[df_parent["Study Instance UID"] == id]
    patient_ids = filtered_parent["Patient ID"].unique()
    if len(patient_ids) != 1:
        print(f"Warning: Multiple patient IDs found for Study Instance UID {id}: {patient_ids}")
    else:
        patient_id = patient_ids[0]
        
    t2_id = filtered_parent[filtered_parent["Series Description"].str.contains("t2", case=False, na=False)]["Series Instance UID"].iloc[0]
    filtered_df = df[df["Series Instance UID (MRI)"] == t2_id]
    assert filtered_df["Patient Number"].unique()[0] == patient_id, f"Mismatch: Patient ID from parent manifest ({patient_id}) does not match Patient Number in biopsy data ({filtered_df['Patient Number'].unique()[0]}) for Study Instance UID {id}"
    if filtered_df["Series Instance UID (US)"].unique().shape[0] == 1:
        assert filtered_df["PSA (ng/mL)"].unique().shape[0] == 1, f"Expected 1 unique PSA value for Study Instance UID {id}, but found {filtered_df['PSA (ng/mL)'].unique().shape[0]}"
        assert filtered_df["Prostate Volume (CC)"].unique().shape[0] == 1, f"Expected 1 unique Volume value for Study Instance UID {id}, but found {filtered_df['Volume (cc)'].unique().shape[0]}"
        gg = (filtered_df['Primary Gleason'].fillna(0) + filtered_df['Secondary Gleason'].fillna(0)).max()
        temp = {}
        temp["image"] = file
        temp["psa"] = [filtered_df["PSA (ng/mL)"].unique()[0], filtered_df["Prostate Volume (CC)"].unique()[0] ]
        temp["dwi"] = os.path.join("/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/DWI_registered", file)
        temp["adc"] = os.path.join("/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/ADC_clipped", file)
        temp["heatmap"] = os.path.join("/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/heatmaps", file)
        temp["mask"] = os.path.join("/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/prostate_mask", file)
        temp["smooth_mask"] = os.path.join("/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/smooth_prostate_mask", file)
        temp["label"] = 1.0 if gg>=7 else 0.0
        test_list.append(temp)
    else:
        exclude.append(id)
len(exclude)

51

In [8]:
test_data = {"test": test_list}
with open("dataset/tcia_test_data.json", "w") as f:
    json.dump(test_data, f, indent=4)

In [9]:
len(test_list)

285

In [10]:
labs = [i['label'] for  i in test_list ]
np.unique(np.array(labs), return_counts=True)


(array([0., 1.]), array([130, 155]))